In [ ]:
# Import, drive and paths

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final"
)

# Mount Google Drive first if it is not already mounted.
if not PROJECT_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")

    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"Project directory not found: {PROJECT_DIR}"
        )

print("Project directory:", PROJECT_DIR)
print("\nSearching for CSV files...\n")

csv_files = sorted(PROJECT_DIR.rglob("*.csv"))

if not csv_files:
    print("No CSV files found inside the project directory.")
else:
    for i, csv_path in enumerate(csv_files, start=1):
        print(f"{i}. {csv_path.relative_to(PROJECT_DIR)}")

print("\nSearch completed.")

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final

Searching for CSV files...

1. Stage4_Segmentation/segmentation_manifest.csv
2. betel_leaf_stratified_split.csv

Search completed.


In [ ]:
# Stage 4.4.1 — Inspect the existing split CSV

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final"
)

SPLIT_CSV = PROJECT_DIR / "betel_leaf_stratified_split.csv"

if not SPLIT_CSV.exists():
    raise FileNotFoundError(
        f"Split CSV not found: {SPLIT_CSV}"
    )

split_df = pd.read_csv(SPLIT_CSV)


print("SPLIT CSV INSPECTION")


print("\nCSV path:")
print(SPLIT_CSV)

print("\nDataset shape:")
print(split_df.shape)

print("\nColumn names:")
print(split_df.columns.tolist())

print("\nFirst 5 rows:")
print(split_df.head().to_string(index=False))

print("\nMissing values per column:")
print(split_df.isnull().sum().to_string())

print("\nUnique values per column:")
for column in split_df.columns:
    print(f"\n{column}:")
    print(split_df[column].value_counts(dropna=False).to_string())

print("\nInspection completed. No files were modified.")

SPLIT CSV INSPECTION

CSV path:
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/betel_leaf_stratified_split.csv

Dataset shape:
(720, 4)

Column names:
['filename', 'path', 'class', 'split']

First 5 rows:
                            filename                                                                                                          path     class split
Bacterial_Leaf_Spot_Disease(105).jpg /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Betel Leaf/bacterial/Bacterial_Leaf_Spot_Disease(105).jpg bacterial  test
Bacterial_Leaf_Spot_Disease(106).jpg /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Betel Leaf/bacterial/Bacterial_Leaf_Spot_Disease(106).jpg bacterial  test
Bacterial_Leaf_Spot_Disease(111).jpg /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Betel Leaf/bacterial/Bacterial_Leaf_Spot_Disease(111).jpg bacterial  test
Bacterial_Leaf_Spot_Disease(114).jpg /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Betel Leaf/bacterial/Bacterial_Leaf_Spot_Disease(

In [ ]:
# Stage 4.4.1 — Split and image-path verification
# cell 3

from pathlib import Path


print("SPLIT AND PATH VERIFICATION")


# 1. Records per split
print("\nImage counts per split:")
print(
    split_df["split"]
    .value_counts()
    .sort_index()
    .to_string()
)

# 2. Class distribution within each split
print("\nClass distribution by split:")
print(
    pd.crosstab(
        split_df["split"],
        split_df["class"],
        margins=True
    ).to_string()
)

# 3. Check duplicate filenames and paths
print("\nDuplicate checks:")
print("Duplicate filenames:", split_df["filename"].duplicated().sum())
print("Duplicate paths:", split_df["path"].duplicated().sum())

# 4. Check whether each recorded image path exists
path_exists = split_df["path"].map(
    lambda p: Path(p).is_file()
)

print("\nImage-path verification:")
print("Existing image files:", int(path_exists.sum()))
print("Missing image files:", int((~path_exists).sum()))

if (~path_exists).any():
    print("\nExamples of missing paths:")
    print(
        split_df.loc[~path_exists, ["filename", "path", "class", "split"]]
        .head(10)
        .to_string(index=False)
    )

# 5. Check expected split and class labels
print("\nUnique split labels:", sorted(split_df["split"].unique()))
print("Unique class labels:", sorted(split_df["class"].unique()))

print("\nVerification cell completed.")

SPLIT AND PATH VERIFICATION

Image counts per split:
split
test          108
train         504
validation    108

Class distribution by split:
class       bacterial  fungal  healthy  All
split                                      
test               37      34       37  108
train             171     160      173  504
validation         37      34       37  108
All               245     228      247  720

Duplicate checks:
Duplicate filenames: 0
Duplicate paths: 0

Image-path verification:
Existing image files: 720
Missing image files: 0

Unique split labels: ['test', 'train', 'validation']
Unique class labels: ['bacterial', 'fungal', 'healthy']

Verification cell completed.


In [ ]:
# Stage 4.4.2 — Match split records to annotations and masks

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final"
)

ANNOTATION_DIR = PROJECT_DIR / "Stage3_Annotation" / "Annotations"
LEAF_MASK_DIR = PROJECT_DIR / "Stage3_Annotation" / "Masks" / "leaf"
LESION_MASK_DIR = PROJECT_DIR / "Stage3_Annotation" / "Masks" / "lesion"

# Build the manifest from the existing split CSV.
manifest = split_df.copy()

manifest["annotation_path"] = manifest["filename"].map(
    lambda name: ANNOTATION_DIR / f"{Path(name).stem}.json"
)

manifest["leaf_mask_path"] = manifest["filename"].map(
    lambda name: LEAF_MASK_DIR / f"{Path(name).stem}.png"
)

manifest["lesion_mask_path"] = manifest["filename"].map(
    lambda name: LESION_MASK_DIR / f"{Path(name).stem}.png"
)

manifest["annotation_exists"] = manifest["annotation_path"].map(
    lambda p: p.is_file()
)

manifest["leaf_mask_exists"] = manifest["leaf_mask_path"].map(
    lambda p: p.is_file()
)

manifest["lesion_mask_exists"] = manifest["lesion_mask_path"].map(
    lambda p: p.is_file()
)


print("ANNOTATION AND MASK COVERAGE")


print("\nOverall file counts:")
print("Split records:", len(manifest))
print("Annotations found:", int(manifest["annotation_exists"].sum()))
print("Leaf masks found:", int(manifest["leaf_mask_exists"].sum()))
print("Lesion masks found:", int(manifest["lesion_mask_exists"].sum()))

print("\nCoverage by split:")
coverage = manifest.groupby("split").agg(
    images=("filename", "size"),
    annotations=("annotation_exists", "sum"),
    leaf_masks=("leaf_mask_exists", "sum"),
    lesion_masks=("lesion_mask_exists", "sum")
)
print(coverage.to_string())

print("\nCoverage by class:")
coverage_class = manifest.groupby("class").agg(
    images=("filename", "size"),
    annotations=("annotation_exists", "sum"),
    leaf_masks=("leaf_mask_exists", "sum"),
    lesion_masks=("lesion_mask_exists", "sum")
)
print(coverage_class.to_string())

# Show any records with missing targets.
missing = manifest[
    ~(
        manifest["annotation_exists"]
        & manifest["leaf_mask_exists"]
        & manifest["lesion_mask_exists"]
    )
]

print("\nRecords with one or more missing files:", len(missing))

if len(missing):
    print(
        missing[
            ["filename", "class", "split",
             "annotation_exists", "leaf_mask_exists",
             "lesion_mask_exists"]
        ].head(20).to_string(index=False)
    )

print("\nCoverage check completed. No files were modified.")

ANNOTATION AND MASK COVERAGE

Overall file counts:
Split records: 720
Annotations found: 720
Leaf masks found: 720
Lesion masks found: 720

Coverage by split:
            images  annotations  leaf_masks  lesion_masks
split                                                    
test           108          108         108           108
train          504          504         504           504
validation     108          108         108           108

Coverage by class:
           images  annotations  leaf_masks  lesion_masks
class                                                   
bacterial     245          245         245           245
fungal        228          228         228           228
healthy       247          247         247           247

Records with one or more missing files: 0

Coverage check completed. No files were modified.


In [ ]:
#3 Project Path and manifest loading
import pandas as pd
import os

PROJECT_ROOT = "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final"

MANIFEST_PATH = os.path.join(
    PROJECT_ROOT,
    "Stage4_Segmentation",
    "segmentation_manifest.csv"
)

segmentation_manifest = pd.read_csv(MANIFEST_PATH)

print("Manifest loaded:", os.path.exists(MANIFEST_PATH))
print("Dataset shape:", segmentation_manifest.shape)
print("Split counts:")
print(segmentation_manifest["split"].value_counts())

Manifest loaded: True
Dataset shape: (720, 12)
Split counts:
split
train         504
test          108
validation    108
Name: count, dtype: int64


In [ ]:
# Stage 4.5.4 — Common preprocessing implementation/function

import numpy as np
import torch
from PIL import Image, ImageOps
from torchvision import transforms

TARGET_SIZE = 512

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

image_normalize = transforms.Normalize(
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD
)


def resize_and_pad(image, target_size, interpolation, fill):
    """
    Preserve aspect ratio, resize, and center-pad to a square.
    """

    width, height = image.size

    scale = min(
        target_size / width,
        target_size / height
    )

    new_width = max(1, round(width * scale))
    new_height = max(1, round(height * scale))

    image = image.resize(
        (new_width, new_height),
        resample=interpolation
    )

    canvas = Image.new(
        image.mode,
        (target_size, target_size),
        color=fill
    )

    left = (target_size - new_width) // 2
    top = (target_size - new_height) // 2

    canvas.paste(image, (left, top))

    return canvas


def preprocess_sample(image_path, leaf_mask_path, lesion_mask_path):
    """
    Apply identical spatial preprocessing to image and masks.
    """

    # Load EXIF-corrected RGB image
    with Image.open(image_path) as img:
        image = ImageOps.exif_transpose(img).convert("RGB")

    # Load binary masks
    with Image.open(leaf_mask_path) as img:
        leaf_mask = img.convert("L")

    with Image.open(lesion_mask_path) as img:
        lesion_mask = img.convert("L")

    # Verify original spatial alignment
    if image.size != leaf_mask.size or image.size != lesion_mask.size:
        raise ValueError(
            f"Image-mask dimension mismatch: {image_path}"
        )

    # Resize and pad RGB image
    image = resize_and_pad(
        image,
        TARGET_SIZE,
        Image.Resampling.BILINEAR,
        (0, 0, 0)
    )

    # Resize and pad masks using nearest-neighbor
    leaf_mask = resize_and_pad(
        leaf_mask,
        TARGET_SIZE,
        Image.Resampling.NEAREST,
        0
    )

    lesion_mask = resize_and_pad(
        lesion_mask,
        TARGET_SIZE,
        Image.Resampling.NEAREST,
        0
    )

    # Convert image to tensor and normalize
    image_tensor = transforms.ToTensor()(image)
    image_tensor = image_normalize(image_tensor)

    # Convert masks to binary float tensors
    leaf_array = np.array(leaf_mask) > 0
    lesion_array = np.array(lesion_mask) > 0

    leaf_tensor = torch.from_numpy(
        leaf_array.astype(np.float32)
    ).unsqueeze(0)

    lesion_tensor = torch.from_numpy(
        lesion_array.astype(np.float32)
    ).unsqueeze(0)

    return image_tensor, leaf_tensor, lesion_tensor


print("Common preprocessing function defined.")
print("Target resolution:", TARGET_SIZE, "x", TARGET_SIZE)
print("Image interpolation: Bilinear")
print("Mask interpolation: Nearest-neighbor")

Common preprocessing function defined.
Target resolution: 512 x 512
Image interpolation: Bilinear
Mask interpolation: Nearest-neighbor


In [ ]:
# Stage 4.6.1 — Common segmentation augmentation function

import random
import torch
import torchvision.transforms.functional as TF

from torchvision.transforms import InterpolationMode
from PIL import Image

TARGET_SIZE = 512

def augment_segmentation_sample(image, leaf_mask, lesion_mask):
    """
    Apply identical geometric transformations to image and masks.
    Apply photometric transformations to RGB image only.

    Inputs:
        image: RGB PIL image
        leaf_mask: binary PIL mask
        lesion_mask: binary PIL mask
    """

    # 1. Random horizontal flip
    if random.random() < 0.5:
        image = TF.hflip(image)
        leaf_mask = TF.hflip(leaf_mask)
        lesion_mask = TF.hflip(lesion_mask)

    # 2. Random vertical flip
    if random.random() < 0.5:
        image = TF.vflip(image)
        leaf_mask = TF.vflip(leaf_mask)
        lesion_mask = TF.vflip(lesion_mask)

    # 3. Random rotation: -15 to +15 degrees
    angle = random.uniform(-15, 15)

    image = TF.rotate(
        image,
        angle,
        interpolation=InterpolationMode.BILINEAR,
        fill=0
    )

    leaf_mask = TF.rotate(
        leaf_mask,
        angle,
        interpolation=InterpolationMode.NEAREST,
        fill=0
    )

    lesion_mask = TF.rotate(
        lesion_mask,
        angle,
        interpolation=InterpolationMode.NEAREST,
        fill=0
    )

    # 4. Random affine transformation: scale and translation
    scale = random.uniform(0.9, 1.1)

    max_translation = int(TARGET_SIZE * 0.05)

    translate_x = random.randint(
        -max_translation,
        max_translation
    )

    translate_y = random.randint(
        -max_translation,
        max_translation
    )

    image = TF.affine(
        image,
        angle=0,
        translate=[translate_x, translate_y],
        scale=scale,
        shear=[0, 0],
        interpolation=InterpolationMode.BILINEAR,
        fill=0
    )

    leaf_mask = TF.affine(
        leaf_mask,
        angle=0,
        translate=[translate_x, translate_y],
        scale=scale,
        shear=[0, 0],
        interpolation=InterpolationMode.NEAREST,
        fill=0
    )

    lesion_mask = TF.affine(
        lesion_mask,
        angle=0,
        translate=[translate_x, translate_y],
        scale=scale,
        shear=[0, 0],
        interpolation=InterpolationMode.NEAREST,
        fill=0
    )

    # 5. Mild photometric augmentation: image only
    brightness = random.uniform(0.85, 1.15)
    contrast = random.uniform(0.85, 1.15)
    saturation = random.uniform(0.90, 1.10)

    image = TF.adjust_brightness(image, brightness)
    image = TF.adjust_contrast(image, contrast)
    image = TF.adjust_saturation(image, saturation)

    return image, leaf_mask, lesion_mask


print("Segmentation augmentation function defined.")

Segmentation augmentation function defined.


In [ ]:
# Stage 4.6.3 — PyTorch segmentation Dataset

import torch
import pandas as pd
import numpy as np

from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Disease class mapping (metadata for later evaluation)
CLASS_TO_IDX = {
    "bacterial": 0,
    "fungal": 1,
    "healthy": 2
}


class BetelLeafSegmentationDataset(Dataset):

    def __init__(self, dataframe, augment=False):

        self.dataframe = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Load EXIF-corrected RGB image
        with Image.open(row["image_path"]) as img:
            image = ImageOps.exif_transpose(img).convert("RGB")

        # Load masks
        with Image.open(row["leaf_mask_path"]) as img:
            leaf_mask = img.convert("L")

        with Image.open(row["lesion_mask_path"]) as img:
            lesion_mask = img.convert("L")

        # Check original alignment
        if (
            image.size != leaf_mask.size
            or image.size != lesion_mask.size
        ):
            raise ValueError(
                f"Image-mask dimension mismatch: {row['filename']}"
            )

        # Common resizing and padding
        image = resize_and_pad(
            image,
            TARGET_SIZE,
            Image.Resampling.BILINEAR,
            (0, 0, 0)
        )

        leaf_mask = resize_and_pad(
            leaf_mask,
            TARGET_SIZE,
            Image.Resampling.NEAREST,
            0
        )

        lesion_mask = resize_and_pad(
            lesion_mask,
            TARGET_SIZE,
            Image.Resampling.NEAREST,
            0
        )

        # Training augmentation only
        if self.augment:

            image, leaf_mask, lesion_mask = (
                augment_segmentation_sample(
                    image,
                    leaf_mask,
                    lesion_mask
                )
            )

        # Convert RGB image to tensor and normalize
        image_tensor = transforms.ToTensor()(image)
        image_tensor = image_normalize(image_tensor)

        # Convert masks to binary tensors
        leaf_array = np.array(leaf_mask) > 0
        lesion_array = np.array(lesion_mask) > 0

        leaf_tensor = torch.from_numpy(
            leaf_array.astype(np.float32)
        ).unsqueeze(0)

        lesion_tensor = torch.from_numpy(
            lesion_array.astype(np.float32)
        ).unsqueeze(0)

        return {
            "image": image_tensor,
            "leaf_mask": leaf_tensor,
            "lesion_mask": lesion_tensor,
            "class_label": torch.tensor(
                CLASS_TO_IDX[row["class"]],
                dtype=torch.long
            ),
            "filename": row["filename"]
        }


print("Betel leaf segmentation Dataset class defined.")

Betel leaf segmentation Dataset class defined.


In [ ]:
# Stage 4.6.3 — Create train, validation and test DataLoaders

BATCH_SIZE = 4
NUM_WORKERS = 0

# Separate the existing fixed partitions
train_df = segmentation_manifest[
    segmentation_manifest["split"] == "train"
].copy()

val_df = segmentation_manifest[
    segmentation_manifest["split"] == "validation"
].copy()

test_df = segmentation_manifest[
    segmentation_manifest["split"] == "test"
].copy()

# Create datasets
train_dataset = BetelLeafSegmentationDataset(
    train_df,
    augment=True
)

val_dataset = BetelLeafSegmentationDataset(
    val_df,
    augment=False
)

test_dataset = BetelLeafSegmentationDataset(
    test_df,
    augment=False
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

print("Segmentation DataLoaders created successfully.")

print("\nDataset sizes:")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nNumber of batches:")
print("Training:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

print("\nBatch size:", BATCH_SIZE)
print("Input resolution:", TARGET_SIZE)

Segmentation DataLoaders created successfully.

Dataset sizes:
Training: 504
Validation: 108
Test: 108

Number of batches:
Training: 126
Validation: 27
Test: 27

Batch size: 4
Input resolution: 512


In [ ]:
# Stage 4.6.3 — Verify DataLoader output

batch = next(iter(train_loader))

print("Training batch verification")

print("\nImage batch shape:")
print(batch["image"].shape)

print("\nLeaf mask batch shape:")
print(batch["leaf_mask"].shape)

print("\nLesion mask batch shape:")
print(batch["lesion_mask"].shape)

print("\nClass labels:")
print(batch["class_label"])

print("\nImage values finite:")
print(torch.isfinite(batch["image"]).all().item())

print("\nLeaf mask values:")
print(torch.unique(batch["leaf_mask"]).tolist())

print("\nLesion mask values:")
print(torch.unique(batch["lesion_mask"]).tolist())

print("\nFilenames in batch:")
print(batch["filename"])

Training batch verification

Image batch shape:
torch.Size([4, 3, 512, 512])

Leaf mask batch shape:
torch.Size([4, 1, 512, 512])

Lesion mask batch shape:
torch.Size([4, 1, 512, 512])

Class labels:
tensor([2, 0, 2, 1])

Image values finite:
True

Leaf mask values:
[0.0, 1.0]

Lesion mask values:
[0.0, 1.0]

Filenames in batch:
['Healthy_Leaf(179).jpg', 'Bacterial_Leaf_Spot_Disease(58).jpg', 'Healthy_Leaf(166).jpg', 'Fungal_Brown_Spot_Disease(187).jpg']


In [ ]:

import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F


# --------------------------------------------------
# Convolution Block (unchanged)
# --------------------------------------------------

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# --------------------------------------------------
# EfficientNet-B0 U-Net
# --------------------------------------------------

class EfficientNetB0_UNet(nn.Module):

    def __init__(self, pretrained=True):
        super().__init__()

        # Load pretrained EfficientNet-B0
        weights = (
            models.EfficientNet_B0_Weights.DEFAULT
            if pretrained else None
        )

        backbone = models.efficientnet_b0(weights=weights)

        self.encoder = backbone.features

        # Decoder
        self.center = ConvBlock(1280, 256)

        self.decoder4 = ConvBlock(256 + 112, 256)
        self.decoder3 = ConvBlock(256 + 40, 128)
        self.decoder2 = ConvBlock(128 + 24, 64)
        self.decoder1 = ConvBlock(64 + 32, 64)

        # Final segmentation layer
        self.final_conv = nn.Sequential(
            ConvBlock(64, 32),
            nn.Conv2d(32, 2, kernel_size=1)
        )

    def forward(self, x):

        input_size = x.shape[2:]

        # Encoder feature extraction

        e1 = self.encoder[0](x)       # 32 channels, 256x256

        x = self.encoder[1](e1)
        e2 = self.encoder[2](x)       # 24 channels, 128x128

        e3 = self.encoder[3](e2)      # 40 channels, 64x64

        x = self.encoder[4](e3)
        e4 = self.encoder[5](x)       # 112 channels, 32x32

        x = self.encoder[6](e4)
        x = self.encoder[7](x)
        e5 = self.encoder[8](x)       # 1280 channels, 16x16

        # Decoder

        center = self.center(e5)

        d4 = F.interpolate(
            center, size=e4.shape[2:],
            mode="bilinear", align_corners=False
        )
        d4 = self.decoder4(torch.cat([d4, e4], dim=1))

        d3 = F.interpolate(
            d4, size=e3.shape[2:],
            mode="bilinear", align_corners=False
        )
        d3 = self.decoder3(torch.cat([d3, e3], dim=1))

        d2 = F.interpolate(
            d3, size=e2.shape[2:],
            mode="bilinear", align_corners=False
        )
        d2 = self.decoder2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(
            d2, size=e1.shape[2:],
            mode="bilinear", align_corners=False
        )
        d1 = self.decoder1(torch.cat([d1, e1], dim=1))

        # Restore original input resolution
        output = F.interpolate(
            d1,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return self.final_conv(output)


In [ ]:
# Set computation device

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

Using device: cuda


In [ ]:
# Initialize and verify EfficientNet-B0 U-Net

model = EfficientNetB0_UNet(pretrained=True).to(DEVICE)

dummy_input = torch.randn(2, 3, 512, 512).to(DEVICE)

model.eval()

with torch.no_grad():
    dummy_output = model(dummy_input)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Input shape:", dummy_input.shape)
print("Output shape:", dummy_output.shape)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Output finite:", torch.isfinite(dummy_output).all().item())

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 159MB/s]


Input shape: torch.Size([2, 3, 512, 512])
Output shape: torch.Size([2, 2, 512, 512])
Total parameters: 9,720,126
Trainable parameters: 9,720,126
Output finite: True


In [ ]:
assert dummy_output.shape == (2, 2, 512, 512), \
    f"Expected output shape (2, 2, 512, 512), got {dummy_output.shape}"

print("Output channel 0: Leaf")
print("Output channel 1: Lesion")

Output channel 0: Leaf
Output channel 1: Lesion


In [ ]:
import torch
import torch.nn as nn

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.view(probabilities.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probabilities * targets).sum(dim=1)

        dice_score = (
            (2.0 * intersection + self.smooth) /
            (probabilities.sum(dim=1) + targets.sum(dim=1) + self.smooth)
        )

        dice_loss = 1.0 - dice_score.mean()

        total_loss = (
            self.bce_weight * bce_loss +
            self.dice_weight * dice_loss
        )

        return total_loss

In [ ]:
# Initialize separate loss criteria
leaf_criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

lesion_criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

model.eval()

with torch.no_grad():
    batch = next(iter(train_loader))

    images = batch["image"].to(DEVICE)
    leaf_targets = batch["leaf_mask"].to(DEVICE)
    lesion_targets = batch["lesion_mask"].to(DEVICE)

    outputs = model(images)

    leaf_logits = outputs[:, 0:1]
    lesion_logits = outputs[:, 1:2]

    leaf_loss = leaf_criterion(leaf_logits, leaf_targets)
    lesion_loss = lesion_criterion(lesion_logits, lesion_targets)

    total_loss = 0.5 * leaf_loss + 0.5 * lesion_loss

print("Model output shape:", outputs.shape)
print("Leaf predictions shape:", leaf_logits.shape)
print("Leaf targets shape:", leaf_targets.shape)
print("Lesion predictions shape:", lesion_logits.shape)
print("Lesion targets shape:", lesion_targets.shape)

print("Leaf loss:", leaf_loss.item())
print("Lesion loss:", lesion_loss.item())
print("Total loss:", total_loss.item())
print("Loss is finite:", torch.isfinite(total_loss).item())

Model output shape: torch.Size([4, 2, 512, 512])
Leaf predictions shape: torch.Size([4, 1, 512, 512])
Leaf targets shape: torch.Size([4, 1, 512, 512])
Lesion predictions shape: torch.Size([4, 1, 512, 512])
Lesion targets shape: torch.Size([4, 1, 512, 512])
Leaf loss: 0.5851525068283081
Lesion loss: 0.8234903216362
Total loss: 0.7043213844299316
Loss is finite: True


In [ ]:
import torch.optim as optim

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40
PATIENCE = 8

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

best_val_loss = float("inf")
epochs_without_improvement = 0

print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Scheduler: ReduceLROnPlateau")

Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.0001
Maximum epochs: 40
Early stopping patience: 8
Scheduler: ReduceLROnPlateau


In [ ]:
import numpy as np

def calculate_global_segmentation_metrics(
    model, loader, device, threshold=0.5
):
    model.eval()

    # Lesion metrics (diseased images only)
    lesion_TP = 0
    lesion_FP = 0
    lesion_FN = 0

    # Leaf metrics (all images)
    leaf_TP = 0
    leaf_FP = 0
    leaf_FN = 0

    healthy_fp_rates = []
    healthy_images_with_fp = 0
    healthy_count = 0

    with torch.no_grad():
        for batch in loader:

            images = batch["image"].to(device)
            lesion_targets = batch["lesion_mask"].to(device)
            leaf_targets = batch["leaf_mask"].to(device)
            labels = batch["class_label"]

            outputs = model(images)

            # Separate the two output channels
            leaf_probabilities = torch.sigmoid(outputs[:, 0:1])
            lesion_probabilities = torch.sigmoid(outputs[:, 1:2])

            leaf_predictions = (
                leaf_probabilities >= threshold
            ).float()

            lesion_predictions = (
                lesion_probabilities >= threshold
            ).float()

            for i in range(images.size(0)):

                # --------------------------------
                # Leaf segmentation metrics
                # --------------------------------
                leaf_pred = leaf_predictions[i]
                leaf_target = leaf_targets[i]

                leaf_TP += (leaf_pred * leaf_target).sum().item()
                leaf_FP += (leaf_pred * (1 - leaf_target)).sum().item()
                leaf_FN += ((1 - leaf_pred) * leaf_target).sum().item()

                # --------------------------------
                # Lesion segmentation metrics
                # --------------------------------
                lesion_pred = lesion_predictions[i]
                lesion_target = lesion_targets[i]

                if labels[i].item() != CLASS_TO_IDX["healthy"]:

                    lesion_TP += (
                        lesion_pred * lesion_target
                    ).sum().item()

                    lesion_FP += (
                        lesion_pred * (1 - lesion_target)
                    ).sum().item()

                    lesion_FN += (
                        (1 - lesion_pred) * lesion_target
                    ).sum().item()

                else:
                    healthy_count += 1

                    # Preserve original methodology:
                    # measure false-positive lesion pixels
                    # only inside the ground-truth leaf.
                    leaf_pixels = leaf_targets[i].sum().item()

                    false_positive_area = (
                        lesion_pred * leaf_targets[i]
                    ).sum().item()

                    fp_rate = (
                        100 * false_positive_area / leaf_pixels
                        if leaf_pixels > 0 else 0
                    )

                    healthy_fp_rates.append(fp_rate)

                    if false_positive_area > 0:
                        healthy_images_with_fp += 1

    # --------------------------------
    # Leaf metrics
    # --------------------------------
    leaf_precision = (
        leaf_TP / (leaf_TP + leaf_FP)
        if leaf_TP + leaf_FP > 0 else 0
    )

    leaf_recall = (
        leaf_TP / (leaf_TP + leaf_FN)
        if leaf_TP + leaf_FN > 0 else 0
    )

    leaf_dice = (
        2 * leaf_TP / (2 * leaf_TP + leaf_FP + leaf_FN)
        if 2 * leaf_TP + leaf_FP + leaf_FN > 0 else 0
    )

    leaf_iou = (
        leaf_TP / (leaf_TP + leaf_FP + leaf_FN)
        if leaf_TP + leaf_FP + leaf_FN > 0 else 0
    )

    # --------------------------------
    # Lesion metrics
    # --------------------------------
    lesion_precision = (
        lesion_TP / (lesion_TP + lesion_FP)
        if lesion_TP + lesion_FP > 0 else 0
    )

    lesion_recall = (
        lesion_TP / (lesion_TP + lesion_FN)
        if lesion_TP + lesion_FN > 0 else 0
    )

    lesion_dice = (
        2 * lesion_TP /
        (2 * lesion_TP + lesion_FP + lesion_FN)
        if 2 * lesion_TP + lesion_FP + lesion_FN > 0 else 0
    )

    lesion_iou = (
        lesion_TP / (lesion_TP + lesion_FP + lesion_FN)
        if lesion_TP + lesion_FP + lesion_FN > 0 else 0
    )

    return {
        "threshold": threshold,

        "leaf_dice": leaf_dice,
        "leaf_iou": leaf_iou,
        "leaf_precision": leaf_precision,
        "leaf_recall": leaf_recall,

        "diseased_dice": lesion_dice,
        "diseased_iou": lesion_iou,
        "diseased_precision": lesion_precision,
        "diseased_recall": lesion_recall,

        "healthy_mean_FP_area_percent": (
            np.mean(healthy_fp_rates) if healthy_fp_rates else 0
        ),
        "healthy_images_with_FP": healthy_images_with_fp,
        "healthy_images": healthy_count
    }

In [ ]:
def train_one_epoch(
    model, loader, leaf_criterion, lesion_criterion,
    optimizer, device
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        images = batch["image"].to(device)
        leaf_targets = batch["leaf_mask"].to(device)
        lesion_targets = batch["lesion_mask"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)

        # Separate output channels
        leaf_logits = outputs[:, 0:1]
        lesion_logits = outputs[:, 1:2]

        # Calculate individual losses
        leaf_loss = leaf_criterion(leaf_logits, leaf_targets)
        lesion_loss = lesion_criterion(lesion_logits, lesion_targets)

        # Equal-weight joint loss
        loss = 0.5 * leaf_loss + 0.5 * lesion_loss

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


def evaluate_model(
    model, loader, leaf_criterion, lesion_criterion, device
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            leaf_targets = batch["leaf_mask"].to(device)
            lesion_targets = batch["lesion_mask"].to(device)

            outputs = model(images)

            # Separate output channels
            leaf_logits = outputs[:, 0:1]
            lesion_logits = outputs[:, 1:2]

            # Calculate individual losses
            leaf_loss = leaf_criterion(leaf_logits, leaf_targets)
            lesion_loss = lesion_criterion(lesion_logits, lesion_targets)

            # Equal-weight joint loss
            loss = 0.5 * leaf_loss + 0.5 * lesion_loss

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / total_samples

In [ ]:
import os
import time
import json
import pandas as pd
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_DIR = os.path.join(
    PROJECT_ROOT,
    "Stage4_Segmentation",
    "Experiment_4B_Joint_BCEDice"
)

CHECKPOINT_DIR = os.path.join(EXPERIMENT_DIR, "Checkpoints")
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, "Results")
FIGURES_DIR = os.path.join(EXPERIMENT_DIR, "Figures")

for folder in [CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(folder, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "efficientnet_b0_unet_best.pth"
)

HISTORY_PATH = os.path.join(
    RESULTS_DIR,
    "efficientnet_b0_training_history.csv"
)

NUM_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

leaf_criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

lesion_criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("Device:", DEVICE)
print("Experiment directory:", EXPERIMENT_DIR)
print("Training configuration initialized.")

Device: cuda
Experiment directory: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage4_Segmentation/Experiment_4B_Joint_BCEDice
Training configuration initialized.


In [ ]:
# Initialize training history

history = {
    "epoch": [],

    "train_loss": [],
    "val_loss": [],

    # Leaf segmentation metrics
    "val_leaf_dice": [],
    "val_leaf_iou": [],
    "val_leaf_precision": [],
    "val_leaf_recall": [],

    # Lesion segmentation metrics
    "val_dice": [],
    "val_iou": [],
    "val_precision": [],
    "val_recall": [],

    # Healthy false-positive evaluation
    "healthy_mean_FP_area_percent": [],
    "healthy_images_with_FP": [],

    "learning_rate": []
}

best_val_loss = float("inf")
epochs_without_improvement = 0

print("Training history initialized.")

Training history initialized.


In [ ]:
# Repair missing history entries for epoch 1

history["healthy_mean_FP_area_percent"].append(healthy_fp)
history["healthy_images_with_FP"].append(healthy_fp_images)

# Save corrected history
pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

print("History successfully repaired and saved.")
print(pd.DataFrame(history).round(5))

NameError: name 'healthy_fp' is not defined

In [ ]:
# Stage 4A — Joint Leaf + Lesion Segmentation Training Loop

start_time = time.time()

# Start a completely new experiment from epoch 1
for epoch in range(NUM_EPOCHS):

    # ------------------------------------------
    # 1. Training
    # ------------------------------------------

    train_loss = train_one_epoch(
        model,
        train_loader,
        leaf_criterion,
        lesion_criterion,
        optimizer,
        DEVICE
    )

    # ------------------------------------------
    # 2. Validation loss
    # ------------------------------------------

    val_loss = evaluate_model(
        model,
        val_loader,
        leaf_criterion,
        lesion_criterion,
        DEVICE
    )

    # ------------------------------------------
    # 3. Validation segmentation metrics
    # ------------------------------------------

    val_metrics = calculate_global_segmentation_metrics(
        model,
        val_loader,
        DEVICE,
        threshold=0.5
    )

    # Leaf metrics
    val_leaf_dice = val_metrics["leaf_dice"]
    val_leaf_iou = val_metrics["leaf_iou"]
    val_leaf_precision = val_metrics["leaf_precision"]
    val_leaf_recall = val_metrics["leaf_recall"]

    # Lesion metrics
    val_dice = val_metrics["diseased_dice"]
    val_iou = val_metrics["diseased_iou"]
    val_precision = val_metrics["diseased_precision"]
    val_recall = val_metrics["diseased_recall"]

    # Healthy false-positive metrics
    healthy_fp = val_metrics["healthy_mean_FP_area_percent"]
    healthy_fp_images = val_metrics["healthy_images_with_FP"]

    # ------------------------------------------
    # 4. Learning rate scheduler
    # ------------------------------------------

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # ------------------------------------------
    # 5. Update training history
    # ------------------------------------------

    history["epoch"].append(epoch + 1)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Leaf metrics
    history["val_leaf_dice"].append(val_leaf_dice)
    history["val_leaf_iou"].append(val_leaf_iou)
    history["val_leaf_precision"].append(val_leaf_precision)
    history["val_leaf_recall"].append(val_leaf_recall)

    # Lesion metrics
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)

    # Healthy false-positive metrics
    history["healthy_mean_FP_area_percent"].append(healthy_fp)
    history["healthy_images_with_FP"].append(healthy_fp_images)

    history["learning_rate"].append(current_lr)

    # ------------------------------------------
    # 6. Checkpoint selection
    # ------------------------------------------

    improved = val_loss < best_val_loss

    if improved:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss
        }, BEST_MODEL_PATH)

        status = "Best model saved"

    else:

        epochs_without_improvement += 1

        status = (
            f"No improvement: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    # ------------------------------------------
    # 7. Save history after every epoch
    # ------------------------------------------

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    # ------------------------------------------
    # 8. Display epoch results
    # ------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Leaf Dice: {val_leaf_dice:.4f} | "
        f"Leaf IoU: {val_leaf_iou:.4f} | "
        f"Lesion Dice: {val_dice:.4f} | "
        f"Lesion IoU: {val_iou:.4f} | "
        f"Lesion Precision: {val_precision:.4f} | "
        f"Lesion Recall: {val_recall:.4f} | "
        f"Healthy FP Area: {healthy_fp:.5f}% | "
        f"Healthy Images with FP: {healthy_fp_images} | "
        f"LR: {current_lr:.6f} | {status}",
        flush=True
    )

    # ------------------------------------------
    # 9. Early stopping
    # ------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print(f"Early stopping at epoch {epoch + 1}.")
        break


# ------------------------------------------
# 10. Training summary
# ------------------------------------------

elapsed_time = (time.time() - start_time) / 60

print("\nJoint segmentation training completed.")
print("Best validation loss:", round(best_val_loss, 5))
print("Total recorded epochs:", len(history["epoch"]))
print("Training time (minutes):", round(elapsed_time, 2))
print("Checkpoint saved:", os.path.exists(BEST_MODEL_PATH))
print("History saved:", os.path.exists(HISTORY_PATH))

Epoch [1/40] | Train Loss: 0.5223 | Val Loss: 0.4602 | Leaf Dice: 0.9752 | Leaf IoU: 0.9516 | Lesion Dice: 0.6293 | Lesion IoU: 0.4591 | Lesion Precision: 0.4935 | Lesion Recall: 0.8680 | Healthy FP Area: 0.21182% | Healthy Images with FP: 37 | LR: 0.000100 | Best model saved
Epoch [2/40] | Train Loss: 0.4398 | Val Loss: 0.4134 | Leaf Dice: 0.9841 | Leaf IoU: 0.9686 | Lesion Dice: 0.8237 | Lesion IoU: 0.7002 | Lesion Precision: 0.7579 | Lesion Recall: 0.9019 | Healthy FP Area: 0.06621% | Healthy Images with FP: 13 | LR: 0.000100 | Best model saved
Epoch [3/40] | Train Loss: 0.4024 | Val Loss: 0.3765 | Leaf Dice: 0.9897 | Leaf IoU: 0.9796 | Lesion Dice: 0.8517 | Lesion IoU: 0.7416 | Lesion Precision: 0.8095 | Lesion Recall: 0.8984 | Healthy FP Area: 0.00338% | Healthy Images with FP: 6 | LR: 0.000100 | Best model saved
Epoch [4/40] | Train Loss: 0.3730 | Val Loss: 0.3549 | Leaf Dice: 0.9904 | Leaf IoU: 0.9810 | Lesion Dice: 0.8456 | Lesion IoU: 0.7325 | Lesion Precision: 0.8958 | Lesion

In [ ]:
experiment_config = {
    "architecture": "EfficientNet-B0 U-Net",
    "encoder_weights": "ImageNet pretrained",
    "input_resolution": "512x512",
    "batch_size": 4,
    "loss": "BCE + Dice",
    "bce_weight": 0.5,
    "dice_weight": 0.5,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "maximum_epochs": NUM_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "scheduler": "ReduceLROnPlateau",
    "selection_criterion": "Lowest validation loss",
    "evaluation_threshold": 0.5
}

CONFIG_PATH = os.path.join(
    RESULTS_DIR,
    "efficientnet_b0_experiment_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(experiment_config, f, indent=4)

print("Experiment configuration saved:", CONFIG_PATH)